In [1]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

c:\Users\Dell\anaconda3\envs\slm_rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
current_directory = os.getcwd()
custom_model_path = os.path.join(current_directory, "AI_MODELS")

print(f"📍 Model will be saved to: {custom_model_path}")
print("   (You can delete this folder manually when you are done)")

📍 Model will be saved to: c:\Users\Dell\Desktop\Ayoub\MyWork\SLM_RAG\notebooks\AI_MODELS
   (You can delete this folder manually when you are done)


In [3]:
model_name = "WiroAI/OpenR1-Qwen-7B-French"

# 1. Configure 8-bit (Perfect for 12GB VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

# 2. Load Tokenizer (Saved to custom folder)
print("\n⬇️ Downloading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name, 
    cache_dir=custom_model_path  # <--- SAVES IT LOCALLY
)

# 3. Load Model (Saved to custom folder)
print("\n⬇️ Downloading Model (This may take time, ~8GB)...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=custom_model_path, # <--- SAVES IT LOCALLY
    torch_dtype=torch.float16
)

print("\n✅ Model loaded successfully!")


⬇️ Downloading Tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!



⬇️ Downloading Model (This may take time, ~8GB)...


Loading checkpoint shards: 100%|██████████| 4/4 [00:13<00:00,  3.43s/it]



✅ Model loaded successfully!


In [4]:
# ==========================================
# 1. SETUP: DATA & CONTEXT
# ==========================================
import json

# Your specific query data
test_data = {
            "id": 9,
            "question": "Quel est le montant de la franchise sur le revenu pour un revenu mensuel de 5000 frs ?",
            "retrieved_contexts": [
                {
                    "source": "Aide sociale et lutte contre la précarité - Règlement - RASLP - 01-01-2025 - xx-xx-xxxx (1).pdf",
                    "type": "Full Section (Merged)",
                    "path": "Chapitre IX      Aide d'urgence et aide ponctuelle / Section 1            Prestations d'aide d'urgence / Art. 67      Prestations à caractère incitatif",
                    "content": "1  Pour  les  personnes  qui  sont  exceptionnellement  au  bénéfice  d'une  autorisation  de  travail,  une  franchise mensuelle par personne est accordée sur le revenu provenant d'une activité lucrative en fonction des critères suivants :\njusqu'à 10 heures de travail mensuelles : 650 francs;\nde 11 à 39 heures de travail mensuelles : 765 francs;\nde 40 à 79 heures de travail mensuelles : 900 francs;\nde 80 à 119 heures de travail mensuelles : 1 050 francs;\ndès 120 heures de travail mensuelles : 1 250 francs.\n2  Pour l'apprentie ou l'apprenti majeur, une franchise mensuelle d'un montant équivalent au salaire est accordée mais au maximum de 1 250 francs.\n3  S'agissant des personnes mineures, aucun revenu provenant de l'activité lucrative n'est pris en compte dans le calcul du droit aux prestations d'aide financière."
                },
                {
                    "source": "Aide sociale et lutte contre la précarité - Règlement - RASLP - 01-01-2025 - xx-xx-xxxx (1).pdf",
                    "type": "Full Section (Merged)",
                    "path": "Chapitre II       Conditions et mode de calcul des prestations d'aide financière / Section 3            Prestations à caractère incitatif / Art. 16      Plafond par groupe familial",
                    "content": "Le montant mensuel de la franchise sur le revenu provenant d'une activité lucrative, au sens de l'article 34, alinéa 2, lettre h, de la loi, accordé au groupe familial, ne peut dépasser 1 200 francs."
                },
                {
                    "source": "Aide sociale et lutte contre la précarité - Règlement - RASLP - 01-01-2025 - xx-xx-xxxx (1).pdf",
                    "type": "Full Section (Merged)",
                    "path": "Chapitre II       Conditions et mode de calcul des prestations d'aide financière / Section 3            Prestations à caractère incitatif / Art. 14      Franchise sur le revenu provenant d'une activité lucrative",
                    "content": "1  En application de l'article 34, alinéa 2, lettre h, de la loi, une franchise mensuelle sur le revenu provenant d'une activité lucrative est accordée aux personnes âgées de 18 ans révolus ou plus.\n2  Cette franchise s'élève :\nà 100% jusqu'à 300 francs nets; et\nà 15% du revenu additionnel net.\n3  La présente disposition est applicable par analogie aux indemnités obtenues dans le cadre d'une activité bénévole."
                },
                {
                    "source": "Aide sociale et lutte contre la précarité - Règlement - RASLP - 01-01-2025 - xx-xx-xxxx (1).pdf",
                    "type": "Full Section (Merged)",
                    "path": "Chapitre II       Conditions et mode de calcul des prestations d'aide financière / Section 3            Prestations à caractère incitatif / Art. 15      Franchise sur le salaire d'apprentissage ou de préapprentissage de l'enfant mineur ou majeur membre du groupe familial",
                    "content": "En application de l'article  34,  alinéa  2,  lettre  f,  de  la  loi,  le  montant  mensuel  de  la  franchise  sur  le  salaire d'apprentissage  ou  de  préapprentissage  de  l'enfant  mineur  ou  majeur  jusqu'à  25 ans  révolus,  membre  du groupe familial, s'élève :\ndurant la première année :\n1° à 100% jusqu'à 600 francs nets, et\n2° à 50% du revenu additionnel net;\ndurant la deuxième année :\n1° à 100% jusqu'à 750 francs nets, et\n2° à 50% du revenu additionnel net;\ndurant la troisième année :\n1° à 100% jusqu'à 900 francs nets, et\n2° à 50% du revenu additionnel net;\ndurant la quatrième année :\n1° à 100% jusqu'à 1 100 francs nets, et\n2° à 50% du revenu additionnel net."
                },
                {
                    "source": "Aide sociale et lutte contre la précarité - Règlement - RASLP - 01-01-2025 - xx-xx-xxxx (1).pdf",
                    "type": "Full Section (Merged)",
                    "path": "Chapitre II       Conditions et mode de calcul des prestations d'aide financière / Section 2            Montants destinés à la couverture des besoins de base / Art. 5        Forfait mensuel pour l'entretien",
                    "content": "1  Le montant mensuel du forfait pour l'entretien au sens de l'article 31, alinéa 2, lettre a, de la loi s'élève à 1 031 francs pour une personne. Ce montant est multiplié par :\n1,53 s'il s'agit de 2 personnes;\n1,86 s'il s'agit de 3 personnes;\n2,19 s'il s'agit de 4 personnes;\n2,52 s'il s'agit de 5 personnes;\n0,28 par personne supplémentaire au-delà de 5 personnes.\nLe résultat est arrondi au franc supérieur.\n2  Le forfait mensuel pour l'entretien est destiné à couvrir les besoins suivants :\nalimentation;\nhabillement;\nconsommation d'énergie, sans les charges locatives;\nentretien du ménage;\nachats de menus articles courants;\nfrais de santé (tels que médicaments achetés sans ordonnance), sans franchise ni quote-part;\ntransport;\ncommunications à distance, Internet, radio/télévision;\nloisirs et formation;\nsoins corporels;\néquipement personnel (tel que fournitures de bureau);\ndivers."
                }
            ]
        }

# Function to format context into a string
def format_context_for_llm(json_list):
    formatted_text = ""
    for item in json_list:
        formatted_text += f"--- SOURCE START ---\n"
        formatted_text += f"Path: {item['path']}\n"
        formatted_text += f"Content: {item['content']}\n"
        formatted_text += f"--- SOURCE END ---\n\n"
    return formatted_text

# ==========================================
# 2. SETUP: YOUR EXACT PROMPT TEMPLATE
# ==========================================
def create_strict_prompt(user_question, context_str):
    return f"""
    <system_role>
    Vous êtes un Moteur de Calcul Juridique expert (RASLP). Votre rôle n'est pas de générer du texte, mais d'exécuter une logique conditionnelle stricte sur le Contexte fourni.
    </system_role>

    <context_data>
    {context_str}
    </context_data>

    <execution_algorithm>
    Vous DEVEZ suivre cet algorithme en 6 étapes pour garantir la justesse juridique.

    1. **FILTRAGE DU RÉGIME (Le "Pare-feu Anti-Urgence") :**
       - **Analyse :** La question contient-elle les mots "Urgence", "Asile", "NEM" ou "Réfugié" ?
       - **RÈGLE STRICTE :** - SI NON : Vous avez l'INTERDICTION formelle d'utiliser toute donnée provenant du "Chapitre IX", de l'"Aide d'urgence" ou des "Articles 63 à 74". Ces données sont toxiques pour une demande standard. Considérez-les comme invisibles.
         - SI OUI : Utilisez prioritairement ces sections.
       - *Correctif Q9/Q11/Q17 : Cela empêche de piocher les franchises du barème d'urgence.*

    2. **MAPPAGE ET DÉTECTION D'ENTITÉS :**
       - Scannez la question pour des statuts déclencheurs d'exception :
         - "Hospitalisé" / "Clinique" -> Chercher "Séjour thérapeutique" ou "Établissement" (Art 35).
         - "Étudiant Haute École" -> Chercher la règle spécifique "Formation" ou "Art 40".
         - "Apprenti" -> Chercher "Contrat d'apprentissage" (Art 15).
       - *Correctif Q15/Q23 : Force le modèle à chercher l'article spécifique (ex: Art 35/40) même si l'Art 5 (Général) est présent.*

    3. **VERIFICATION DES BORNES D'ÂGE (La règle de "Réversion") :**
       - **RÈGLE :** Si un statut spécifique (ex: Apprenti, Étudiant) est identifié, vérifiez immédiatement si le texte impose une LIMITE D'ÂGE (ex: "jusqu'à 25 ans").
       - **ACTION :** Si l'utilisateur dépasse cette limite (ex: 27 ans), vous DEVEZ ANNULER le statut spécifique et appliquer le régime "Adulte / Activité Lucrative Ordinaire" (Art 14).
       - *Correctif Q22/Q30 : Un apprenti de 27 ans ne doit pas utiliser le barème apprenti limité à 25 ans.*

    4. **VALIDATION DES DONNÉES (Brut/Net) :**
       - Si le texte exige du "Net" et que l'utilisateur donne du "Brut" (sans formule de conversion disponible), signalez-le.
       - Par défaut, pour l'aide sociale genevoise, les revenus sont considérés en NET sauf mention contraire.

    5. **CALCUL ET EXTRAPOLATION :**
       - Appliquez la formule mathématique trouvée dans le BON article (sélectionné à l'étape 1 et 2).
       - Pour les familles nombreuses hors tableau : Appliquez la formule "Par personne supplémentaire".

    6. **APPLICATION DES PLAFONDS :**
       - Vérifiez toujours s'il existe un "Montant Maximum" global pour la catégorie (Loyer, Franchise Famille). Le Plafond gagne toujours sur le calcul.

    </execution_algorithm>

    <examples>
    
    *Cas 1 : Évitement du piège de l'Urgence*
    Utilisateur : "Franchise pour salaire 4000 ?"
    Contexte : [Art 14 (Ordinaire): 300+15%] ... [Art 67 (Urgence): Tableau fixe 1250]
    Raisonnement : Mot "Urgence" absent. Art 67 IGNORÉ. Calcul basé sur Art 14.
    Réponse : 855

    *Cas 2 : Réversion d'Âge (Le piège de l'apprenti vieux)*
    Utilisateur : "Franchise apprenti de 28 ans gagnant 2000 ?"
    Contexte : [Art 15 : Apprentis jusqu'à 25 ans = Franchise X] [Art 14 : Adultes = Franchise Y]
    Raisonnement : Age 28 > Limite 25. Art 15 invalidé. Bascule sur Art 14 (Adulte).
    Réponse : [Calcul basé sur Art 14]

    *Cas 3 : Priorité Spécifique*
    Utilisateur : "Forfait pour une personne à l'hôpital ?"
    Contexte : [Art 5 : Personne à domicile = 1000] [Art 35 : Personne en établissement = 200]
    Raisonnement : "Hôpital" détecté. Art 5 ignoré. Art 35 appliqué.
    Réponse : 200

    </examples>

    <response_format>
    ÉTAPE 1 - LOGIQUE D'EXÉCUTION :
    - Régime Appliqué : [Standard OU Urgence] (Justification : mot-clé présent/absent)
    - Article Sélectionné : [ex: Art 14, car Art 67 exclu] ou [Art 14 car âge > 25]
    - Données Utilisées : [Revenu, Loyer, Taille famille...]
    - Calcul Détaillé : [Formule complète]

    ÉTAPE 2 - RÉPONSE FINALE :
    [Juste le chiffre et l'unité, ou une phrase courte si non calculable]
    </response_format>

    <user_question>
    {user_question}
    </user_question>
    """

# ==========================================
# 3. GENERATION
# ==========================================
from matplotlib.style import context
import torch

# Prepare inputs
context_str = format_context_for_llm(test_data["retrieved_contexts"])
full_prompt = create_strict_prompt(test_data["question"], context_str)

messages = [
    {"role": "system", "content": "Please reason step-by-step."}, 
    {"role": "user", "content": full_prompt}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

print(f"❓ Question: {test_data['question']}")
print("🧠 Model is thinking (using OpenR1-Qwen-7B-French)...")

with torch.no_grad():
    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=2048,
        temperature=0.3, # Low temp for math precision
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("\n" + "="*40)
print("🤖 MODEL OUTPUT")
print("="*40 + "\n")
print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


❓ Question: Quel est le montant de la franchise sur le revenu pour un revenu mensuel de 5000 frs ?
🧠 Model is thinking (using OpenR1-Qwen-7B-French)...

🤖 MODEL OUTPUT

<|begin_of_thought|>
D'accord, abordons cette question étape par étape. Tout d'abord, je dois vérifier si le contexte contient des informations pertinentes. La question porte sur la franchise sur le revenu pour un revenu mensuel de 5000 CHF. Je vais examiner les sources fournies.

En regardant les sources, il y a plusieurs articles liés aux franchises sur le revenu provenant d'une activité lucratif. Les articles clés ici sont probablement ceux qui traitent des franchises générales pour les personnes âgées de 18 ans et plus (Art. 14) et celles spécifiques à certains groupes comme les apprentis ou les personnes âgées de moins de 25 ans.

La question ne mentionne rien concernant l'urgence, l'asile, NEM ou les réfugiés, donc le "Pare-feu Anti-Urgence" ne devrait pas entrer en jeu. Cela signifie que nous pouvons utiliser les

In [ ]:
# ==========================================
# 1. SETUP: DATA & CONTEXT
# ==========================================
import json

# Your specific query data
test_data = {
            "id": 31,
            "question": "Le salaire d'apprentissage d'un enfant de 17 ans en 1ère année est de 2000 frs.\nQuelle est la franchise à appliquer sur ce salaire ?",
            "retrieved_contexts": [
                {
                    "source": "Aide sociale et lutte contre la précarité - Règlement - RASLP - 01-01-2025 - xx-xx-xxxx (1).pdf",
                    "type": "Full Section (Merged)",
                    "path": "Chapitre II       Conditions et mode de calcul des prestations d'aide financière / Section 3            Prestations à caractère incitatif / Art. 15      Franchise sur le salaire d'apprentissage ou de préapprentissage de l'enfant mineur ou majeur membre du groupe familial",
                    "content": "En application de l'article  34,  alinéa  2,  lettre  f,  de  la  loi,  le  montant  mensuel  de  la  franchise  sur  le  salaire d'apprentissage  ou  de  préapprentissage  de  l'enfant  mineur  ou  majeur  jusqu'à  25 ans  révolus,  membre  du groupe familial, s'élève :\ndurant la première année :\n1° à 100% jusqu'à 600 francs nets, et\n2° à 50% du revenu additionnel net;\ndurant la deuxième année :\n1° à 100% jusqu'à 750 francs nets, et\n2° à 50% du revenu additionnel net;\ndurant la troisième année :\n1° à 100% jusqu'à 900 francs nets, et\n2° à 50% du revenu additionnel net;\ndurant la quatrième année :\n1° à 100% jusqu'à 1 100 francs nets, et\n2° à 50% du revenu additionnel net."
                },
                {
                    "source": "Aide sociale et lutte contre la précarité - Règlement - RASLP - 01-01-2025 - xx-xx-xxxx (1).pdf",
                    "type": "Full Section (Merged)",
                    "path": "Chapitre IX      Aide d'urgence et aide ponctuelle / Section 1            Prestations d'aide d'urgence / Art. 67      Prestations à caractère incitatif",
                    "content": "1  Pour  les  personnes  qui  sont  exceptionnellement  au  bénéfice  d'une  autorisation  de  travail,  une  franchise mensuelle par personne est accordée sur le revenu provenant d'une activité lucrative en fonction des critères suivants :\njusqu'à 10 heures de travail mensuelles : 650 francs;\nde 11 à 39 heures de travail mensuelles : 765 francs;\nde 40 à 79 heures de travail mensuelles : 900 francs;\nde 80 à 119 heures de travail mensuelles : 1 050 francs;\ndès 120 heures de travail mensuelles : 1 250 francs.\n2  Pour l'apprentie ou l'apprenti majeur, une franchise mensuelle d'un montant équivalent au salaire est accordée mais au maximum de 1 250 francs.\n3  S'agissant des personnes mineures, aucun revenu provenant de l'activité lucrative n'est pris en compte dans le calcul du droit aux prestations d'aide financière."
                },
                {
                    "source": "Aide sociale et lutte contre la précarité - Règlement - RASLP - 01-01-2025 - xx-xx-xxxx (1).pdf",
                    "type": "Full Section (Merged)",
                    "path": "Chapitre II       Conditions et mode de calcul des prestations d'aide financière / Section 3            Prestations à caractère incitatif / Art. 14      Franchise sur le revenu provenant d'une activité lucrative",
                    "content": "1  En application de l'article 34, alinéa 2, lettre h, de la loi, une franchise mensuelle sur le revenu provenant d'une activité lucrative est accordée aux personnes âgées de 18 ans révolus ou plus.\n2  Cette franchise s'élève :\nà 100% jusqu'à 300 francs nets; et\nà 15% du revenu additionnel net.\n3  La présente disposition est applicable par analogie aux indemnités obtenues dans le cadre d'une activité bénévole."
                },
                {
                    "source": "Aide sociale et lutte contre la précarité - Règlement - RASLP - 01-01-2025 - xx-xx-xxxx (1).pdf",
                    "type": "Full Section (Merged)",
                    "path": "Chapitre II       Conditions et mode de calcul des prestations d'aide financière / Section 3            Prestations à caractère incitatif / Art. 16      Plafond par groupe familial",
                    "content": "Le montant mensuel de la franchise sur le revenu provenant d'une activité lucrative, au sens de l'article 34, alinéa 2, lettre h, de la loi, accordé au groupe familial, ne peut dépasser 1 200 francs."
                },
                {
                    "source": "Aide sociale et lutte contre la précarité - Règlement - RASLP - 01-01-2025 - xx-xx-xxxx (1).pdf",
                    "type": "Full Section (Merged)",
                    "path": "Chapitre IV      Prestations d'aide financière relatives aux situations particulières / Section 2            Personnes exerçant une activité lucrative indépendante / Art. 44      Revenus pris en compte pour les personnes exerçant une activité indépendante",
                    "content": "Les revenus pris en compte sont déterminés conformément à l'article 34 de la loi, sous réserve de la franchise sur  le  revenu  provenant  d'une  activité  lucrative,  au  sens  de  l'article  34,  alinéa  2,  lettre  h,  de  la  loi,  qui  ne s'applique pas."
                }
            ]
        }

# Function to format context into a string
def format_context_for_llm(json_list):
    formatted_text = ""
    for item in json_list:
        formatted_text += f"--- SOURCE START ---\n"
        formatted_text += f"Path: {item['path']}\n"
        formatted_text += f"Content: {item['content']}\n"
        formatted_text += f"--- SOURCE END ---\n\n"
    return formatted_text

# ==========================================
# 2. SETUP: YOUR EXACT PROMPT TEMPLATE
# ==========================================
def create_strict_prompt(user_question, context_str):
    return f"""
    <system_role>
    Vous êtes un Moteur de Calcul Juridique expert. Votre objectif est d'extraire la logique du Contexte et de l'exécuter pour répondre à la Question Utilisateur.
    Vous suivez les règles strictement. Vous ne résumez pas ; vous calculez.
    </system_role>

    <context_data>
    {context_str}
    </context_data>

    <execution_algorithm>
    Pour répondre à la question, vous DEVEZ suivre strictement cet algorithme en 6 étapes. Le non-respect de l'ordre entraînera une pénalité.

    1. **VÉRIFICATION DU PÉRIMÈTRE (La règle du "Mauvais Livre") :**
       - Les textes juridiques contiennent souvent des régimes différents (ex : "Standard/Général" vs "Urgence/Asile").
       - **RÈGLE :** Si la Question Utilisateur est standard, vous DEVEZ IGNORER les valeurs trouvées dans les sections intitulées "Urgence", "Asile" ou "Mesures exceptionnelles", sauf si l'utilisateur les demande explicitement.
       - *Logique :* Question Générale = Données de la Section Générale uniquement. Ne croisez pas les limites.

    2. **MAPPAGE DES ENTITÉS (La règle des "Synonymes") :**
       - Les termes juridiques varient souvent. Vous devez faire correspondre les termes de l'Utilisateur aux définitions du Texte.
       - **Correspondances courantes :**
         - "Mineur" → Traiter comme "Enfant à charge".
         - "Hospitalisé" → Traiter comme "Séjour thérapeutique".
         - "Famille de N [personnes]" → Traiter comme "Chef de famille + (N-1) Personnes à charge".

    3. **PRIORITÉ DE SPÉCIFICITÉ (La règle de l'"Exception") :**
       - **RÈGLE :** Le statut spécifique l'emporte sur le statut général.
       - **ACTION :** Avant d'appliquer un taux "Adulte Général", recherchez spécifiquement les attributs de l'Utilisateur (ex : "Étudiant", "Apprenti", "Stagiaire").
       - Si une réduction ou une règle spécifique existe pour ce statut (ex : "Les étudiants obtiennent 70 %"), vous DEVEZ l'appliquer.

    4. **VALIDATION DES ENTRÉES (La règle "Brut vs Net") :**
       - Vérifiez les types de variables.
       - **RÈGLE :** Si la formule du texte exige un "Revenu Net" et que l'utilisateur fournit un "Revenu Brut", ne calculez PAS en utilisant le chiffre Brut. À moins qu'une formule de conversion n'existe dans le texte, renvoyez "Non Applicable".

    5. **EXÉCUTION DE LA FORMULE (La règle des "Maths") :**
       - Si le texte définit une méthode (ex : "Base + Majoration" ou "Revenu - Déduction"), vous DEVEZ effectuer le calcul.
       - **Extrapolation Familiale :** Si un tableau s'arrête à la taille N (ex : 4 personnes) mais implique une règle pour "chaque personne supplémentaire", calculez la valeur pour la taille demandée par l'Utilisateur (ex : 5).

    6. **CONTRAINTES GLOBALES (La règle du "Plafond") :**
       - Un calcul n'est jamais terminé tant que vous n'avez pas vérifié les Limites.
       - **RÈGLE :** Scannez TOUT le texte à la recherche de "Maximum" ou "Plafond" s'appliquant à la catégorie spécifique (Famille, Loyer, Aide).
       - *Logique :* Réponse Finale = MIN(Montant Calculé, Montant Plafond).
    </execution_algorithm>

    <examples>
    Voici des exemples comment appliquer correctement la logique :

    *Exemple 1 : La Logique de "Périmètre" (Standard vs Urgence)*
    Utilisateur : "Quelle est la franchise pour un adulte ?"
    Contexte : [Art 14 : Franchise Adulte = 300 frs] [Art 67 (Urgence) : Franchise Adulte = 1250 frs]
    Raisonnement : L'utilisateur n'a PAS demandé l'Aide d'Urgence. Ignorer Art 67. Utiliser Art 14.
    Réponse Finale : 300

    *Exemple 2 : La Logique d'"Exception" (Étudiant vs Adulte)*
    Utilisateur : "Quel est le forfait d'entretien pour un Étudiant ?"
    Contexte : [Art 5 : Entretien de Base = 1000] [Art 40 : Étudiant = 70% de la Base]
    Raisonnement : L'utilisateur est "Étudiant". La Règle Spécifique (Art 40) l'emporte sur la Règle Générale (Art 5).
    Calcul : 1000 * 0.70 = 700.
    Réponse Finale : 700

    *Exemple 3 : La Logique de "Plafond" (Formule vs Plafond)*
    Utilisateur : "Franchise pour un salaire d'apprenti de 2000 ?"
    Contexte : [Art 15 : Franchise = 1300] [Art 16 : Plafond Famille = 1200]
    Raisonnement : Calculé 1300. Trouvé Plafond Global de 1200 à l'Art 16. Le Plafond est plus bas.
    Réponse Finale : 1200
    </examples>

    <response_format>
    Vous DEVEZ utiliser ce format exact. Ne faites pas de paragraphes.

    ÉTAPE 1 - RAISONNEMENT :
    - Périmètre Identifié : [ex : Régime Standard (section Urgence ignorée)]
    - Entité/Statut Mappé : [ex : Mappé "Mineur" vers "Enfant"]
    - Formule Identifiée : [Citer la règle/le texte]
    - Vérification d'Éligibilité : [Succès/Échec selon âge/statut]
    - Étapes de Calcul : [Montrer les maths : ex : Base 500 + Extra 100 = 600]
    - Vérification du Plafond : [Est-ce que 600 > Plafond 550 ? Oui, donc le résultat est 550]

    ÉTAPE 2 - RÉPONSE FINALE :
    [Le chiffre final ou "Non Applicable"]
    </response_format>

    <user_question>
    {user_question}
    </user_question>
    """

# ==========================================
# 3. GENERATION
# ==========================================
from matplotlib.style import context
import torch

# Prepare inputs
context_str = format_context_for_llm(test_data["retrieved_contexts"])
full_prompt = create_strict_prompt(test_data["question"], context_str)

messages = [
    {"role": "system", "content": "Please reason step-by-step."}, 
    {"role": "user", "content": full_prompt}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

print(f"❓ Question: {test_data['question']}")
print("🧠 Model is thinking (using OpenR1-Qwen-7B-French)...")

with torch.no_grad():
    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=2048,
        temperature=0.3, # Low temp for math precision
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("\n" + "="*40)
print("🤖 MODEL OUTPUT")
print("="*40 + "\n")
print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


❓ Question: Le salaire d'apprentissage d'un enfant de 17 ans en 1ère année est de 2000 frs.
Quelle est la franchise à appliquer sur ce salaire ?
🧠 Model is thinking (using OpenR1-Qwen-7B-French)...

🤖 MODEL OUTPUT

<|begin_of_thought|>
D'accord, abordons cette question étape par étape. Tout d'abord, je dois vérifier si la franchise pour apprentissage s'applique ici. L'utilisateur demande une franchise sur un salaire d'apprentissage d'un enfant de 17 ans en première année. Le montant est de 2000 francs.

En regardant les sources fournies, il existe une section spécifique sur la franchise sur le salaire d'apprentissage. L'article 15 mentionne que la franchise est accordée aux enfants de moins de 25 ans, y compris ceux de 25 ans. Cependant, l'utilisateur a spécifié un enfant de 17 ans, ce qui est bien inférieur à 25 ans. Donc, la première condition est remplie.

Maintenant, la franchise est calculée différemment selon la première année. L'article 15 indique que la franchise est de 100 % j

In [ ]:
# ==========================================
# 1. SETUP: DATA & CONTEXT
# ==========================================
import json

# Your specific query data
test_data = {
            "id": 31,
            "question": "Le salaire d'apprentissage d'un enfant de 17 ans en 1ère année est de 2000 frs.\nQuelle est la franchise à appliquer sur ce salaire ?",
            "retrieved_contexts": [
                {
                    "source": "Aide sociale et lutte contre la précarité - Règlement - RASLP - 01-01-2025 - xx-xx-xxxx (1).pdf",
                    "type": "Full Section (Merged)",
                    "path": "Chapitre II       Conditions et mode de calcul des prestations d'aide financière / Section 3            Prestations à caractère incitatif / Art. 15      Franchise sur le salaire d'apprentissage ou de préapprentissage de l'enfant mineur ou majeur membre du groupe familial",
                    "content": "En application de l'article  34,  alinéa  2,  lettre  f,  de  la  loi,  le  montant  mensuel  de  la  franchise  sur  le  salaire d'apprentissage  ou  de  préapprentissage  de  l'enfant  mineur  ou  majeur  jusqu'à  25 ans  révolus,  membre  du groupe familial, s'élève :\ndurant la première année :\n1° à 100% jusqu'à 600 francs nets, et\n2° à 50% du revenu additionnel net;\ndurant la deuxième année :\n1° à 100% jusqu'à 750 francs nets, et\n2° à 50% du revenu additionnel net;\ndurant la troisième année :\n1° à 100% jusqu'à 900 francs nets, et\n2° à 50% du revenu additionnel net;\ndurant la quatrième année :\n1° à 100% jusqu'à 1 100 francs nets, et\n2° à 50% du revenu additionnel net."
                },
                {
                    "source": "Aide sociale et lutte contre la précarité - Règlement - RASLP - 01-01-2025 - xx-xx-xxxx (1).pdf",
                    "type": "Full Section (Merged)",
                    "path": "Chapitre IX      Aide d'urgence et aide ponctuelle / Section 1            Prestations d'aide d'urgence / Art. 67      Prestations à caractère incitatif",
                    "content": "1  Pour  les  personnes  qui  sont  exceptionnellement  au  bénéfice  d'une  autorisation  de  travail,  une  franchise mensuelle par personne est accordée sur le revenu provenant d'une activité lucrative en fonction des critères suivants :\njusqu'à 10 heures de travail mensuelles : 650 francs;\nde 11 à 39 heures de travail mensuelles : 765 francs;\nde 40 à 79 heures de travail mensuelles : 900 francs;\nde 80 à 119 heures de travail mensuelles : 1 050 francs;\ndès 120 heures de travail mensuelles : 1 250 francs.\n2  Pour l'apprentie ou l'apprenti majeur, une franchise mensuelle d'un montant équivalent au salaire est accordée mais au maximum de 1 250 francs.\n3  S'agissant des personnes mineures, aucun revenu provenant de l'activité lucrative n'est pris en compte dans le calcul du droit aux prestations d'aide financière."
                },
                {
                    "source": "Aide sociale et lutte contre la précarité - Règlement - RASLP - 01-01-2025 - xx-xx-xxxx (1).pdf",
                    "type": "Full Section (Merged)",
                    "path": "Chapitre II       Conditions et mode de calcul des prestations d'aide financière / Section 3            Prestations à caractère incitatif / Art. 14      Franchise sur le revenu provenant d'une activité lucrative",
                    "content": "1  En application de l'article 34, alinéa 2, lettre h, de la loi, une franchise mensuelle sur le revenu provenant d'une activité lucrative est accordée aux personnes âgées de 18 ans révolus ou plus.\n2  Cette franchise s'élève :\nà 100% jusqu'à 300 francs nets; et\nà 15% du revenu additionnel net.\n3  La présente disposition est applicable par analogie aux indemnités obtenues dans le cadre d'une activité bénévole."
                },
                {
                    "source": "Aide sociale et lutte contre la précarité - Règlement - RASLP - 01-01-2025 - xx-xx-xxxx (1).pdf",
                    "type": "Full Section (Merged)",
                    "path": "Chapitre II       Conditions et mode de calcul des prestations d'aide financière / Section 3            Prestations à caractère incitatif / Art. 16      Plafond par groupe familial",
                    "content": "Le montant mensuel de la franchise sur le revenu provenant d'une activité lucrative, au sens de l'article 34, alinéa 2, lettre h, de la loi, accordé au groupe familial, ne peut dépasser 1 200 francs."
                },
                {
                    "source": "Aide sociale et lutte contre la précarité - Règlement - RASLP - 01-01-2025 - xx-xx-xxxx (1).pdf",
                    "type": "Full Section (Merged)",
                    "path": "Chapitre IV      Prestations d'aide financière relatives aux situations particulières / Section 2            Personnes exerçant une activité lucrative indépendante / Art. 44      Revenus pris en compte pour les personnes exerçant une activité indépendante",
                    "content": "Les revenus pris en compte sont déterminés conformément à l'article 34 de la loi, sous réserve de la franchise sur  le  revenu  provenant  d'une  activité  lucrative,  au  sens  de  l'article  34,  alinéa  2,  lettre  h,  de  la  loi,  qui  ne s'applique pas."
                }
            ]
        }

# Function to format context into a string
def format_context_for_llm(json_list):
    formatted_text = ""
    for item in json_list:
        formatted_text += f"--- SOURCE START ---\n"
        formatted_text += f"Path: {item['path']}\n"
        formatted_text += f"Content: {item['content']}\n"
        formatted_text += f"--- SOURCE END ---\n\n"
    return formatted_text

# ==========================================
# 2. SETUP: YOUR EXACT PROMPT TEMPLATE
# ==========================================
def create_strict_prompt(user_question, context_str):
    return f"""
    <system_role>
    Vous êtes un Moteur de Calcul Juridique expert. Votre objectif est d'extraire la logique du Contexte et de l'exécuter pour répondre à la Question Utilisateur.
    Vous suivez les règles strictement. Vous ne résumez pas ; vous calculez.
    </system_role>

    <context_data>
    {context_str}
    </context_data>

    <execution_algorithm>
    Pour répondre à la question, vous DEVEZ suivre strictement cet algorithme en 6 étapes. Le non-respect de l'ordre entraînera une pénalité.

    1. **VÉRIFICATION DU PÉRIMÈTRE (La règle du "Mauvais Livre") :**
       - Les textes juridiques contiennent souvent des régimes différents (ex : "Standard/Général" vs "Urgence/Asile").
       - **RÈGLE :** Si la Question Utilisateur est standard, vous DEVEZ IGNORER les valeurs trouvées dans les sections intitulées "Urgence", "Asile" ou "Mesures exceptionnelles", sauf si l'utilisateur les demande explicitement.
       - *Logique :* Question Générale = Données de la Section Générale uniquement. Ne croisez pas les limites.

    2. **MAPPAGE DES ENTITÉS (La règle des "Synonymes") :**
       - Les termes juridiques varient souvent. Vous devez faire correspondre les termes de l'Utilisateur aux définitions du Texte.
       - **Correspondances courantes :**
         - "Mineur" → Traiter comme "Enfant à charge".
         - "Hospitalisé" → Traiter comme "Séjour thérapeutique".
         - "Famille de N [personnes]" → Traiter comme "Chef de famille + (N-1) Personnes à charge".

    3. **PRIORITÉ DE SPÉCIFICITÉ (La règle de l'"Exception") :**
       - **RÈGLE :** Le statut spécifique l'emporte sur le statut général.
       - **ACTION :** Avant d'appliquer un taux "Adulte Général", recherchez spécifiquement les attributs de l'Utilisateur (ex : "Étudiant", "Apprenti", "Stagiaire").
       - Si une réduction ou une règle spécifique existe pour ce statut (ex : "Les étudiants obtiennent 70 %"), vous DEVEZ l'appliquer.

    4. **VALIDATION DES ENTRÉES (La règle "Brut vs Net") :**
       - Vérifiez les types de variables.
       - **RÈGLE :** Si la formule du texte exige un "Revenu Net" et que l'utilisateur fournit un "Revenu Brut", ne calculez PAS en utilisant le chiffre Brut. À moins qu'une formule de conversion n'existe dans le texte, renvoyez "Non Applicable".

    5. **EXÉCUTION DE LA FORMULE (La règle des "Maths") :**
       - Si le texte définit une méthode (ex : "Base + Majoration" ou "Revenu - Déduction"), vous DEVEZ effectuer le calcul.
       - **Extrapolation Familiale :** Si un tableau s'arrête à la taille N (ex : 4 personnes) mais implique une règle pour "chaque personne supplémentaire", calculez la valeur pour la taille demandée par l'Utilisateur (ex : 5).

    6. **CONTRAINTES GLOBALES (La règle du "Plafond") :**
       - Un calcul n'est jamais terminé tant que vous n'avez pas vérifié les Limites.
       - **RÈGLE :** Scannez TOUT le texte à la recherche de "Maximum" ou "Plafond" s'appliquant à la catégorie spécifique (Famille, Loyer, Aide).
       - *Logique :* Réponse Finale = MIN(Montant Calculé, Montant Plafond).
    </execution_algorithm>

    <examples>
    Voici des exemples comment appliquer correctement la logique :

    *Exemple 1 : La Logique de "Périmètre" (Standard vs Urgence)*
    Utilisateur : "Quelle est la franchise pour un adulte ?"
    Contexte : [Art 14 : Franchise Adulte = 300 frs] [Art 67 (Urgence) : Franchise Adulte = 1250 frs]
    Raisonnement : L'utilisateur n'a PAS demandé l'Aide d'Urgence. Ignorer Art 67. Utiliser Art 14.
    Réponse Finale : 300

    *Exemple 2 : La Logique d'"Exception" (Étudiant vs Adulte)*
    Utilisateur : "Quel est le forfait d'entretien pour un Étudiant ?"
    Contexte : [Art 5 : Entretien de Base = 1000] [Art 40 : Étudiant = 70% de la Base]
    Raisonnement : L'utilisateur est "Étudiant". La Règle Spécifique (Art 40) l'emporte sur la Règle Générale (Art 5).
    Calcul : 1000 * 0.70 = 700.
    Réponse Finale : 700

    *Exemple 3 : La Logique de "Plafond" (Formule vs Plafond)*
    Utilisateur : "Franchise pour un salaire d'apprenti de 2000 ?"
    Contexte : [Art 15 : Franchise = 1300] [Art 16 : Plafond Famille = 1200]
    Raisonnement : Calculé 1300. Trouvé Plafond Global de 1200 à l'Art 16. Le Plafond est plus bas.
    Réponse Finale : 1200
    </examples>

    <response_format>
    Vous DEVEZ utiliser ce format exact. Ne faites pas de paragraphes.

    ÉTAPE 1 - RAISONNEMENT :
    - Périmètre Identifié : [ex : Régime Standard (section Urgence ignorée)]
    - Entité/Statut Mappé : [ex : Mappé "Mineur" vers "Enfant"]
    - Formule Identifiée : [Citer la règle/le texte]
    - Vérification d'Éligibilité : [Succès/Échec selon âge/statut]
    - Étapes de Calcul : [Montrer les maths : ex : Base 500 + Extra 100 = 600]
    - Vérification du Plafond : [Est-ce que 600 > Plafond 550 ? Oui, donc le résultat est 550]

    ÉTAPE 2 - RÉPONSE FINALE :
    [Le chiffre final ou "Non Applicable"]
    </response_format>

    <user_question>
    {user_question}
    </user_question>
    """

# ==========================================
# 3. GENERATION
# ==========================================
from matplotlib.style import context
import torch

# Prepare inputs
context_str = format_context_for_llm(test_data["retrieved_contexts"])
full_prompt = create_strict_prompt(test_data["question"], context_str)

messages = [
    {"role": "system", "content": "Please reason step-by-step."}, 
    {"role": "user", "content": full_prompt}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

print(f"❓ Question: {test_data['question']}")
print("🧠 Model is thinking (using OpenR1-Qwen-7B-French)...")

with torch.no_grad():
    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=2048,
        temperature=0.3, # Low temp for math precision
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("\n" + "="*40)
print("🤖 MODEL OUTPUT")
print("="*40 + "\n")
print(response)

❓ Question: Le salaire d'apprentissage d'un enfant de 17 ans en 1ère année est de 2000 frs.
Quelle est la franchise à appliquer sur ce salaire ?
🧠 Model is thinking (using OpenR1-Qwen-7B-French)...

🤖 MODEL OUTPUT

<|begin_of_thought|>
D'accord, abordons cette question étape par étape. Tout d'abord, je dois identifier le contexte pertinent ici. L'utilisateur demande la franchise pour le salaire d'apprentissage d'un enfant de 17 ans en première année, qui est de 2000 francs.

En regardant les données fournies, il y a plusieurs articles mentionnant différentes franchises. L'Article 15 semble pertinent car il traite de la franchise sur le salaire d'apprentissage pour les enfants de moins de 25 ans. Laissez-moi vérifier cela. L'article 15 dit : "En application de l'article 34, alinéa 2, lettre f, de la loi, le montant mensuel de la franchise sur le salaire d'apprentissage..." Il y a ensuite des plages horaires différentes avec des pourcentages.

Maintenant, l'enfant a 17 ans, ce qui est in